# TATN: Panasonic 25 C to CMU eVTOL

This notebook runs the same CMU core-seven protocol twice: (1) a TATN source trained once from the paper's Panasonic 25 C procedure, and (2) the authors' published `pre-25.pt`. CMU SOC references are used only after adaptation for evaluation.

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')
with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check token permission')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import battery_benchmark
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Package:', battery_benchmark.__file__)

## Shared protocol

- Panasonic source: all nine 25 C profiles, sampled at 1 Hz, 90-step non-overlapping windows.
- Self-trained source: paper-layout tensors; Adam, lr 1e-3, batch 64, max 300 epochs with label-safe early stopping.
- Released source: exact `pre-25.pt` and its required released-repository tensor layout.
- CMU adaptation: the same chronological complete-mission prefix used by DARM, reaching at least 8,778 seconds.
- Each CMU cell starts from the unchanged source checkpoint. No adapted state crosses cells.
- CMU target normalization uses only the Panasonic 25 C source ranges.

In [ ]:
import pandas as pd
import torch
from IPython.display import display
from battery_benchmark.utils.config import load_yaml

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
DRIVE_OUTPUT = Path(PATHS['output_root'])
PANASONIC_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
CMU_BASE_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_ns_relabel'
CMU_CONDITION_ROOT = REPO / 'outputs/datasets/cmu_evtol/dynamic_capacity_soc_development_v1_extension4_ns_relabel'
CMU_MANIFEST = REPO / 'data/manifests/cmu_evtol/cmu_evtol_supervised_budget_manifest_v0_1_ns_relabel.csv'
OFFICIAL_PRE25 = DRIVE_OUTPUT / 'reproductions/shen_2023/tatn_layout_ablation/official_pre25/pre-25.pt'
RESULT_ROOT = DRIVE_OUTPUT / 'benchmarks/panasonic_to_cmu_evtol/tatn_ns_relabel_v1'
CELLS = ['VAH01', 'VAH17', 'VAH27', 'VAH30', 'VAH15', 'VAH12', 'VAH11']
SOURCE_EPOCHS, TRANSFER_EPOCHS, BATCH_SIZE = 300, 150, 64
for path in (PANASONIC_ROOT, CMU_BASE_ROOT, CMU_CONDITION_ROOT, CMU_MANIFEST, OFFICIAL_PRE25):
    assert path.exists(), f'Missing required input: {path}'

def run_variant(source_kind, output_name, released_checkpoint=None, scenario='unsupervised', trained_checkpoint=None):
    command = [
        sys.executable, str(REPO / 'scripts/benchmarks/run_tatn_cmu.py'),
        '--panasonic-data-root', str(PANASONIC_ROOT),
        '--cmu-base-root', str(CMU_BASE_ROOT),
        '--cmu-condition-root', str(CMU_CONDITION_ROOT),
        '--cmu-manifest', str(CMU_MANIFEST),
        '--output-root', str(RESULT_ROOT / output_name),
        '--source-kind', source_kind, '--source-epochs', str(SOURCE_EPOCHS),
        '--transfer-epochs', str(TRANSFER_EPOCHS), '--batch-size', str(BATCH_SIZE),
        '--device', 'cuda' if torch.cuda.is_available() else 'cpu', '--resume',
        '--cells', *CELLS,
    ]
    if released_checkpoint is not None:
        command += ['--released-checkpoint', str(released_checkpoint)]
    if trained_checkpoint is not None:
        command += ['--trained-checkpoint', str(trained_checkpoint)]
    if scenario != 'unsupervised':
        command += ['--scenario', scenario]
    subprocess.run(command, check=True)
    result = pd.read_csv(RESULT_ROOT / output_name / 'condition_metrics.csv')
    display(result.pivot(index=['cell_id', 'condition'], columns='method', values=['overall_mae_percent', 'overall_rmse_percent']).round(3))
    return result

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('results:', RESULT_ROOT)

## Result 1 - source trained from the Panasonic 25 C paper protocol

In [ ]:
trained_results = run_variant('trained', 'trained_paper_layout')

## Result 2 - authors' published Panasonic 25 C checkpoint

In [ ]:
released_results = run_variant('released', 'released_pre25', OFFICIAL_PRE25)

## Semisupervised protocol

For each CMU cell, the earliest adaptation mission contributes the first 50% of its SOC reference labels; all other adaptation missions remain unlabelled and use selected pseudolabels. Test missions remain held out. These CMU SOC references are development estimates rather than laboratory ground truth. The two runs below reuse the source checkpoints from Results 1 and 2 and write to separate output directories.

## Result 3 - semisupervised, source trained from the Panasonic 25 C paper protocol

In [ ]:
TRAINED_SOURCE_CHECKPOINT = RESULT_ROOT / 'trained_paper_layout/source_25C.pt'
assert TRAINED_SOURCE_CHECKPOINT.exists(), 'Run Result 1 first to create the trained source checkpoint'
trained_semisupervised_results = run_variant(
    'trained', 'trained_paper_layout_semisupervised',
    scenario='semisupervised', trained_checkpoint=TRAINED_SOURCE_CHECKPOINT,
)

## Result 4 - semisupervised, authors' published Panasonic 25 C checkpoint

In [ ]:
released_semisupervised_results = run_variant(
    'released', 'released_pre25_semisupervised', OFFICIAL_PRE25,
    scenario='semisupervised',
)

## Diagnostics: adaptation training and validation curves

Everything below is read straight back from what `run_tatn_cmu.py` already
wrote under each variant's output directory -- per-cell and top-level
`training_history.csv` (and `source_training_history.csv` when source
pretraining actually ran that invocation) -- nothing here re-runs TATN or
recomputes a metric from scratch. `training_history.csv` has one row per
(cell, epoch) of the adaptation loop; `estimate_loss`/`adversarial_loss`/
`mmd_loss`/`total_loss` are the same values already used for the scheduler
and checkpoint selection. Where the CMU manifest's diagnostic-only
`validation` split had missions for a cell, `adapt()` also probes
`validation_mae`/`validation_rmse` on epoch 1, every `--validation-interval`
epochs (default 10), and the actual last epoch -- this is a READ-ONLY probe
that never fed back into training, pseudo-labeling, the scheduler, early
stopping, or checkpoint selection (see `run_metadata.json`'s
`validation_protocol` note).

In [ ]:
import matplotlib.pyplot as plt

VARIANT_OUTPUT_NAMES = [
    'trained_paper_layout', 'released_pre25',
    'trained_paper_layout_semisupervised', 'released_pre25_semisupervised',
]

def load_training_history(output_name):
    path = RESULT_ROOT / output_name / 'training_history.csv'
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()

for output_name in VARIANT_OUTPUT_NAMES:
    history = load_training_history(output_name)
    if history.empty:
        print(f'No training_history.csv yet for {output_name}')
        continue

    fig, axes = plt.subplots(1, 4, figsize=(20, 4))
    for ax, column in zip(axes, ['estimate_loss', 'adversarial_loss', 'mmd_loss', 'total_loss']):
        for cell_id, group in history.groupby('cell_id'):
            ax.plot(group['epoch'], group[column], marker='.', label=cell_id, alpha=0.6)
        ax.set_title(column)
        ax.set_xlabel('epoch')
    axes[0].legend(fontsize=7)
    fig.suptitle(f'{output_name}: adaptation loss curves per cell')
    plt.tight_layout()
    plt.show()

In [ ]:
for output_name in VARIANT_OUTPUT_NAMES:
    history = load_training_history(output_name)
    if history.empty:
        continue
    evaluated = history.loc[history['validation_evaluated'] == True]
    if evaluated.empty:
        print(f'{output_name}: no validation diagnostics recorded')
        continue

    fig, (mae_ax, rmse_ax) = plt.subplots(1, 2, figsize=(12, 4))
    for cell_id, group in evaluated.groupby('cell_id'):
        mae_ax.plot(group['epoch'], group['validation_mae'], marker='o', label=cell_id, alpha=0.7)
        rmse_ax.plot(group['epoch'], group['validation_rmse'], marker='o', label=cell_id, alpha=0.7)
    mae_ax.set_title(f'{output_name}: diagnostic-only validation MAE per epoch\n(read-only, never used for training/selection)')
    mae_ax.set_xlabel('epoch')
    mae_ax.set_ylabel('validation_mae (percent)')
    rmse_ax.set_title(f'{output_name}: diagnostic-only validation RMSE per epoch')
    rmse_ax.set_xlabel('epoch')
    rmse_ax.set_ylabel('validation_rmse (percent)')
    mae_ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
for output_name in VARIANT_OUTPUT_NAMES:
    history = load_training_history(output_name)
    print(f'--- {output_name} ---')
    if not history.empty:
        time_summary = (
            history.groupby('cell_id')
            .agg(
                total_train_seconds=('runtime_seconds', 'sum'),
                epochs=('runtime_seconds', 'count'),
                total_validation_seconds=('validation_runtime_seconds', 'sum'),
                validation_epochs=('validation_evaluated', 'sum'),
            )
            .round(3)
        )
        display(time_summary)
    else:
        print('No training_history.csv yet')

    source_history_path = RESULT_ROOT / output_name / 'source_training_history.csv'
    if source_history_path.is_file():
        source_history = pd.read_csv(source_history_path)
        print(f'Source pretraining time (once for this variant): {source_history["runtime_seconds"].sum():.3f}s over {len(source_history)} epochs')
    else:
        print('No source_training_history.csv (source was loaded from an existing checkpoint, not trained this run)')

## Diagnostics: predictions and phase-level negative transfer

Everything below is read straight back from what `run_tatn_cmu.py`
already wrote under each variant's output directory --
`test_predictions.parquet` (per-cell, then aggregated at the top
level), `phase_metrics.csv`, `phase_transfer_effects.csv`, and
`cell_transfer_effects.csv` -- nothing here re-runs TATN, recomputes a
prediction, or recomputes a metric from scratch. `test_predictions.
parquet` has one row per raw CMU test-split timestep for both the
'Source only' (frozen source model) and 'TATN' (post-adaptation)
methods, built from the SAME truth/prediction arrays already used for
`condition_metrics.csv` above. `mission_phase` is restricted to
`take_off`/`cruise`/`landing` in the phase tables (the canonical Ns
mapping via `battery_benchmark.benchmarks.cmu_diagnostics`).
`delta_*_percent = TATN - 'Source only'`, and `negative_transfer_*`
flags a positive delta (TATN worse than the frozen source baseline).

In [ ]:
def load_csv_if_present(output_name, filename):
    path = RESULT_ROOT / output_name / filename
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()

for output_name in VARIANT_OUTPUT_NAMES:
    condition_metrics = load_csv_if_present(output_name, 'condition_metrics.csv')
    if condition_metrics.empty:
        print(f'No condition_metrics.csv yet for {output_name}')
        continue
    print(f'--- {output_name}: Source only vs TATN, overall MAE/RMSE per cell ---')
    display(
        condition_metrics.pivot(
            index=['cell_id', 'condition'], columns='method',
            values=['overall_mae_percent', 'overall_rmse_percent'],
        ).round(3)
    )

In [ ]:
for output_name in VARIANT_OUTPUT_NAMES:
    cell_transfer_effects = load_csv_if_present(output_name, 'cell_transfer_effects.csv')
    if cell_transfer_effects.empty:
        print(f'No cell_transfer_effects.csv yet for {output_name}')
        continue
    print(f'--- {output_name}: overall negative-transfer delta (TATN - Source only) ---')
    display(
        cell_transfer_effects[
            ['cell_id', 'condition', 'delta_mae_percent', 'delta_rmse_percent',
             'negative_transfer_mae', 'negative_transfer_rmse']
        ].round(3)
    )
    n_negative = int(cell_transfer_effects['negative_transfer_mae'].sum())
    print(f'{n_negative} of {len(cell_transfer_effects)} cells show negative transfer (MAE)')

In [ ]:
for output_name in VARIANT_OUTPUT_NAMES:
    phase_metrics = load_csv_if_present(output_name, 'phase_metrics.csv')
    if phase_metrics.empty:
        print(f'No phase_metrics.csv yet for {output_name}')
        continue
    print(f'--- {output_name}: Source only vs TATN, three-phase MAE/RMSE ---')
    display(
        phase_metrics.pivot(
            index=['cell_id', 'condition', 'mission_phase'], columns='method',
            values=['mae_percent', 'rmse_percent'],
        ).round(3)
    )

In [ ]:
for output_name in VARIANT_OUTPUT_NAMES:
    phase_transfer_effects = load_csv_if_present(output_name, 'phase_transfer_effects.csv')
    if phase_transfer_effects.empty:
        print(f'No phase_transfer_effects.csv yet for {output_name}')
        continue
    print(f'--- {output_name}: per-phase negative-transfer counts (MAE) ---')
    display(
        phase_transfer_effects.groupby('mission_phase')['negative_transfer_mae']
        .agg(negative_transfer_cells='sum', total_cells='count')
    )
    display(
        phase_transfer_effects[
            ['cell_id', 'condition', 'mission_phase', 'delta_mae_percent',
             'delta_rmse_percent', 'negative_transfer_mae', 'negative_transfer_rmse']
        ].round(3)
    )